In [1]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np

In [2]:
mt5.initialize()

True

In [3]:
SYMBOL       = 'EURUSD'
VOLUME       = 0.1
DEVIATION    = 10
point        = mt5.symbol_info(SYMBOL).point
STOPLOSS     = 250
TAKEPROFIT   = 350

In [4]:
# FUNCTION - PLACE A MARKET ORDER
def market_order(symbol, volume, order_type, sl_pips=None, tp_pips=None, **kwargs):
    """
    Place a market order with dynamically calculated stop loss (SL) and take profit (TP).
    
    Args:
        symbol (str): Trading symbol.
        volume (float): Volume of the trade.
        order_type (str): 'buy' or 'sell'.
        sl_pips (float, optional): Stop loss distance in pips. Default is None.
        tp_pips (float, optional): Take profit distance in pips. Default is None.
        **kwargs: Additional parameters for flexibility.
    
    Returns:
        object: Result of the `mt5.order_send` call.
    """
    tick = mt5.symbol_info_tick(symbol)
    if tick is None:
        print(f"Error: Unable to fetch tick info for symbol {symbol}")
        return None

    # Fetch the symbol's pip size
    symbol_info = mt5.symbol_info(symbol)
    if symbol_info is None:
        print(f"Error: Unable to fetch symbol info for {symbol}")
        return None
    pip_size = symbol_info.point

    order_dict = {'buy': 0, 'sell': 1}
    price_dict = {'buy': tick.ask, 'sell': tick.bid}

    # Calculate SL and TP prices
    sl = None
    tp = None
    if sl_pips is not None:
        sl = price_dict[order_type] - sl_pips * pip_size if order_type == 'buy' else price_dict[order_type] + sl_pips * pip_size
    if tp_pips is not None:
        tp = price_dict[order_type] + tp_pips * pip_size if order_type == 'buy' else price_dict[order_type] - tp_pips * pip_size

    # Construct the request
    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": symbol,
        "volume": volume,
        "type": order_dict[order_type],
        "price": price_dict[order_type],
        "sl": sl,
        "tp": tp,
        "deviation": DEVIATION,
        "magic": 999,
        "comment": "python market order",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_IOC,
    }

    # Send the order request
    order_result = mt5.order_send(request)

    return order_result


# FUNCTION - CLOSE AN OPEN ORDER
def close_order(ticket, symbol):
    positions = mt5.positions_get()

    for pos in positions:
        tick = mt5.symbol_info_tick(pos.symbol)
        type_dict = {0: 1, 1: 0}  # 0 represents buy, 1 represents sell - inverting order_type to close the position
        price_dict = {0: tick.ask, 1: tick.bid}
        
        if pos.ticket == ticket and pos.symbol == symbol:
            request = {
                "action": mt5.TRADE_ACTION_DEAL,
                "position": pos.ticket,
                "symbol": pos.symbol,
                "volume": pos.volume,
                "type": type_dict[pos.type],
                "price": price_dict[pos.type],
                "deviation": DEVIATION,
                "magic": 999,
                "comment": "python close order",
                "type_time": mt5.ORDER_TIME_GTC,
                "type_filling": mt5.ORDER_FILLING_IOC,
            }
            order_result = mt5.order_send(request)
            
            return order_result

    return "Ticket doesn't Exist"


def modify_order(ticket, symbol, sl_pips=None, tp_pips=None):
    """
    Modify an open order's stop loss (SL) and take profit (TP) using pips as input.
    
    Args:
        ticket (int): Ticket number of the order.
        symbol (str): Trading symbol.
        sl_pips (float, optional): New stop loss distance in pips. Default is None.
        tp_pips (float, optional): New take profit distance in pips. Default is None.
    
    Returns:
        object: Result of the `mt5.order_send` call.
    """
    positions = mt5.positions_get(symbol=symbol)

    if positions:
        for pos in positions:
            if pos.ticket == ticket:
                # Get the tick data for calculating SL/TP
                tick = mt5.symbol_info_tick(symbol)
                if tick is None:
                    print(f"Error: Unable to fetch tick info for {symbol}")
                    return None

                pip_size = mt5.symbol_info(symbol).point

                # Calculate new stop loss price
                if sl_pips is not None:
                    if pos.type == mt5.ORDER_TYPE_BUY:  # For Buy position
                        new_sl = pos.price_open - sl_pips * pip_size
                    elif pos.type == mt5.ORDER_TYPE_SELL:  # For Sell position
                        new_sl = pos.price_open + sl_pips * pip_size
                else:
                    new_sl = None

                # Calculate new take profit price
                if tp_pips is not None:
                    if pos.type == mt5.ORDER_TYPE_BUY:  # For Buy position
                        new_tp = pos.price_open + tp_pips * pip_size
                    elif pos.type == mt5.ORDER_TYPE_SELL:  # For Sell position
                        new_tp = pos.price_open - tp_pips * pip_size
                else:
                    new_tp = None

                # Send the modification request
                request = {
                    "action": mt5.TRADE_ACTION_SLTP,
                    "position": pos.ticket,
                    "symbol": pos.symbol,
                    "sl": new_sl,
                    "tp": new_tp,
                    "magic": 999,
                    "comment": "python modify order",
                }
                result = mt5.order_send(request)
                return result

    return f"No position found for ticket {ticket}"



# FUNCTION - GET EXPOSURE
def get_exposure(symbol):
    positions = mt5.positions_get(symbol=symbol)
    if positions:
        pos_df = pd.DataFrame(positions, columns=positions[0]._asdict().keys())
        exposure = pos_df['volume'].sum()
        
        return exposure


In [8]:
market_order("XAUUSD", 0.1, "sell", sl_pips=STOPLOSS, tp_pips=TAKEPROFIT)

OrderSendResult(retcode=10009, deal=669681141, order=847695051, volume=0.1, price=2626.79, bid=2626.79, ask=2626.84, comment='Request executed', request_id=3549624635, retcode_external=0, request=TradeRequest(action=1, magic=999, order=0, symbol='XAUUSD', volume=0.1, price=2626.79, stoplimit=0.0, sl=2629.29, tp=2623.29, deviation=10, type=1, type_filling=1, type_time=0, expiration=0, comment='python market order', position=0, position_by=0))

In [26]:
def calculate_macd(data, short_period=12, long_period=26, signal_period=9):
    """Calculate the MACD and signal line."""
    data['ema_short'] = data['close'].ewm(span=short_period, adjust=False).mean()
    data['ema_long'] = data['close'].ewm(span=long_period, adjust=False).mean()
    data['MACD'] = data['ema_short'] - data['ema_long']
    data['signal_line'] = data['MACD'].ewm(span=signal_period, adjust=False).mean()
    data['MACD_histogram'] = data['MACD'] - data['signal_line']
    return data


def generate_macd_trend(df):
    df['Trend'] = 0

    # Generate buy (1) and sell (-1) Trend
    for i in range(1, len(df)):
        # Bullish crossover (Buy)
        if df.loc[i, 'MACD'] > df.loc[i, 'signal_line']:
            df.loc[i, 'Trend'] = 1

        # Bearish crossover (Sell)
        elif df.loc[i, 'MACD'] <= df.loc[i, 'signal_line']:
            df.loc[i, 'Trend'] = 0

    return df


def generate_macd_signals(df):
        # Initialize the 'signal' column with 0 (no signal)
    df['Signal'] = 0

    # Generate buy (1) and sell (-1) signals
    for i in range(1, len(df)):
        # Bullish crossover (Buy)
        if df.loc[i, 'MACD'] > df.loc[i, 'signal_line'] and df.loc[i - 1, 'MACD'] <= df.loc[i - 1, 'signal_line']:
            df.loc[i, 'Signal'] = 1

        # Bearish crossover (Sell)
        elif df.loc[i, 'MACD'] < df.loc[i, 'signal_line'] and df.loc[i - 1, 'MACD'] >= df.loc[i - 1, 'signal_line']:
            df.loc[i, 'Signal'] = -1

    return df

In [27]:
import MetaTrader5 as mt5
import pandas as pd

def check_crossover(symbol, timeframe, short_period=12, long_period=26, signal_period=9):
    """
    Check if a MACD crossover occurred on the previous candle for the given symbol and timeframe.
    
    Parameters:
        symbol (str): The trading symbol (e.g., "EURUSD").
        timeframe: MetaTrader 5 timeframe (e.g., mt5.TIMEFRAME_M1).
        short_period (int): Short period for the MACD calculation. Default is 12.
        long_period (int): Long period for the MACD calculation. Default is 26.
        signal_period (int): Signal line period for the MACD calculation. Default is 9.
    
    Returns:
        bool: True if a MACD crossover occurred, False otherwise.
    """
    # Ensure MT5 is initialized
    if not mt5.initialize():
        raise Exception("MetaTrader 5 initialization failed.")

    # Get the data for the specified timeframe
    num_candles = 100  # Ensure enough data for MACD calculation
    rates = mt5.copy_rates_from_pos(symbol, timeframe, 1, num_candles)
    
    if rates is None or len(rates) == 0:
        raise Exception(f"Failed to retrieve data for {symbol} on timeframe {timeframe}.")
    
    # Convert rates to DataFrame
    data = pd.DataFrame(rates)
    data['time'] = pd.to_datetime(data['time'], unit='s')
    # data.set_index('time', inplace=True)
    
    # Calculate MACD and signal line
    data = calculate_macd(data, short_period, long_period, signal_period)
    
    # Generate signals
    data = generate_macd_signals(data)
    
    # Check for crossover in the last complete candle (second-to-last row)
    if len(data) < 2:
        return False  # Not enough data to determine
    
    return data.iloc[-1]['Signal'] != 0  # True if there was a signal in the previous candle


def check_trend(symbol, timeframe, short_period=12, long_period=26, signal_period=9):
    """
    Check if a MACD crossover occurred on the previous candle for the given symbol and timeframe.
    
    Parameters:
        symbol (str): The trading symbol (e.g., "EURUSD").
        timeframe: MetaTrader 5 timeframe (e.g., mt5.TIMEFRAME_M1).
        short_period (int): Short period for the MACD calculation. Default is 12.
        long_period (int): Long period for the MACD calculation. Default is 26.
        signal_period (int): Signal line period for the MACD calculation. Default is 9.
    
    Returns:
        bool: True if a MACD crossover occurred, False otherwise.
    """
    # Ensure MT5 is initialized
    if not mt5.initialize():
        raise Exception("MetaTrader 5 initialization failed.")

    # Get the data for the specified timeframe
    num_candles = 100  # Ensure enough data for MACD calculation
    rates = mt5.copy_rates_from_pos(symbol, timeframe, 2, num_candles)
    
    if rates is None or len(rates) == 0:
        raise Exception(f"Failed to retrieve data for {symbol} on timeframe {timeframe}.")
    
    # Convert rates to DataFrame
    data = pd.DataFrame(rates)
    data['time'] = pd.to_datetime(data['time'], unit='s')
    # data.set_index('time', inplace=True)
    
    # Calculate MACD and signal line
    data = calculate_macd(data, short_period, long_period, signal_period)
    
    # Generate signals
    data = generate_macd_trend(data)
    
    # Check for crossover in the last complete candle (second-to-last row)
    if len(data) < 2:
        return False  # Not enough data to determine
    
    return data.iloc[-1]['Trend']  # True if there was a signal in the previous candle


In [28]:
def entry_logic(symbol):
    """
    Determine if an entry signal is valid based on MACD crossovers and trend alignment.
    
    Parameters:
        symbol (str): The trading symbol (e.g., "EURUSD").
    
    Returns:
        bool: True if entry conditions are met, False otherwise.
    """
    try:
        # Check for MACD crossovers
        m5_cross    = check_crossover(symbol, mt5.TIMEFRAME_M5)
        m30_cross   = check_crossover(symbol, mt5.TIMEFRAME_M30)
        h1_cross    = check_crossover(symbol, mt5.TIMEFRAME_H1)

        # Check for trend alignment
        m5_trend    = check_trend(symbol, mt5.TIMEFRAME_M5)
        m30_trend   = check_trend(symbol, mt5.TIMEFRAME_M30)
        h1_trend    = check_trend(symbol, mt5.TIMEFRAME_H1)

        m5_cross = True
        
        # Entry logic
        if m5_cross or m30_cross or h1_cross:
            print(m30_trend)
            if m5_trend == m30_trend == h1_trend:
                return True

        return False  # Default return if conditions are not met

    except Exception as e:
        print(f"Error in entry_logic: {e}")
        return False  # Safely handle errors and return False
    
    
def exit_logic(symbol):
    
    m5_trend    = check_trend(symbol, mt5.TIMEFRAME_M5)
    m15_trend   = check_trend(symbol, mt5.TIMEFRAME_M15)
    m30_trend   = check_trend(symbol, mt5.TIMEFRAME_M30)  
    
    if m5_trend == m15_trend == m30_trend:
        return True

    else: 
        return False
    
def trial_logic(symbol):
    
    m5_trend    = check_trend(symbol, mt5.TIMEFRAME_M5)
    m15_trend   = check_trend(symbol, mt5.TIMEFRAME_M15)
    m30_trend   = check_trend(symbol, mt5.TIMEFRAME_M30)
    
    
    if m5_trend == m15_trend == m30_trend:
        return True

    else: 
        return False
    

In [30]:
entry_logic("EURUSD")

1


True

In [17]:
check_trend("EURUSD", mt5.TIMEFRAME_M5)

0

In [18]:
def has_active_trade(symbol):
    """
    Check if there is an active trade for the given symbol.
    
    Args:
        symbol (str): The trading symbol to check.

    Returns:
        bool: True if there is an active trade for the symbol, False otherwise.
    """
    positions = mt5.positions_get(symbol=symbol)
    return len(positions) > 0 if positions else False


In [19]:
def check_existing_trade(symbol):
    positions = mt5.positions_get(symbol=symbol)
    result = {
        "buy" : False,
        "sell" : False
    }
    for pos in positions:
        if pos.type == mt5.ORDER_TYPE_BUY:
            result['buy'] = True
        elif pos.type == mt5.ORDER_TYPE_SELL:
            result['sell'] = True
            
    return result

In [21]:
import time
import logging
from logging.handlers import TimedRotatingFileHandler
import os

# Globals
previous_tick = None

count = 0

root_dir = SYMBOL

# Log file paths
MAIN_LOG  = rf"{root_dir}\trading_bot.log"
ERROR_LOG = rf"{root_dir}\error.log"
TRADE_LOG = rf"{root_dir}\trade.log"

def check_existing_trade(symbol):
    positions = mt5.positions_get(symbol=symbol)
    result = {
        "buy" : False,
        "sell" : False
    }
    for pos in positions:
        if pos.type == mt5.ORDER_TYPE_BUY:
            result['buy'] = True
        elif pos.type == mt5.ORDER_TYPE_SELL:
            result['sell'] = True
            
    return result
# Initialize loggers
def initialize_loggers():
    global main_logger, error_logger, trade_logger

    # Main logger (deleted every minute)
    main_logger = logging.getLogger("main")
    main_logger.setLevel(logging.INFO)
    main_handler = TimedRotatingFileHandler(MAIN_LOG, when="M", interval=1, backupCount=1)
    main_handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
    main_logger.addHandler(main_handler)

    # Error logger (never deleted)
    error_logger = logging.getLogger("error")
    error_logger.setLevel(logging.ERROR)
    error_handler = logging.FileHandler(ERROR_LOG)
    error_handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
    error_logger.addHandler(error_handler)

    # Trade logger (never deleted)
    trade_logger = logging.getLogger("trade")
    trade_logger.setLevel(logging.INFO)
    trade_handler = logging.FileHandler(TRADE_LOG)
    trade_handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
    trade_logger.addHandler(trade_handler)

# Function to check for a new tick
def check_tick(symbol):
    """
    Check if a new tick has occurred for the given symbol.
    
    Args:
        symbol (str): Trading symbol.
    
    Returns:
        bool: True if a new tick occurred, otherwise False.
    """
    global previous_tick

    tick = mt5.symbol_info_tick(symbol)
    if tick is None:
        main_logger.warning(f"Failed to fetch tick data for {symbol}")
        return False

    if previous_tick is None or tick.ask != previous_tick.ask or tick.bid != previous_tick.bid:
        previous_tick = tick
        return True

    return False

# Main trading loop
def main_trading_loop():
    global previous_tick
    global count
    global TAKEPROFIT
    if not mt5.initialize():
        main_logger.error("MetaTrader5 initialization failed")
        error_logger.error("MetaTrader5 initialization failed")
        return

    main_logger.info("Trading bot started.")

    while True:
        try:
            tick_occurred = check_tick(SYMBOL)

            if tick_occurred:
                main_logger.info("New tick detected.")
                
                # Fetch open positions for the symbol
                positions = mt5.positions_get(symbol=SYMBOL)

                if positions:
                    for position in positions:
                        if exit_logic(SYMBOL):
                            if position.type == check_trend(SYMBOL, mt5.TIMEFRAME_M5):
                                close_order(position.ticket, SYMBOL)
                        if position.profit == 100:
                            main_logger.info(f"Position {position.ticket} profit > 100. Modifying SL to -10 pips.")
                            modify_order(position.ticket, SYMBOL, sl_pips=-10.0, tp_pips=TAKEPROFIT)

                        elif position.profit > (TAKEPROFIT - 50):
                            if trial_logic(SYMBOL):
                                if position.type != check_trend(SYMBOL, mt5.TIMEFRAME_M5):
                                    if count < 2:
                                        main_logger.info(f"Position {position.ticket} nearing TP. Adjusting SL and TP.")
                                        modify_order(
                                            position.ticket,
                                            SYMBOL,
                                            sl_pips=-100.0,
                                            tp_pips=TAKEPROFIT + 50,
                                        )

                                        TAKEPROFIT = TAKEPROFIT + 50
                                        count += 1
                else:
                    main_logger.info(f"No open positions found for {SYMBOL}.")

                # Entry logic
                if entry_logic(SYMBOL):
                    main_logger.info(f"Entry logic triggered for {SYMBOL}.")
                    result = check_existing_trade(SYMBOL)
                    
                    
                    if check_trend(SYMBOL, mt5.TIMEFRAME_M5) == 1:
                        if result['buy'] == False:
                            main_logger.info("Uptrend detected. Placing buy order.")
                            market_order(SYMBOL, VOLUME, "buy", sl_pips=STOPLOSS, tp_pips=TAKEPROFIT) 
                            trade_logger.info("Uptrend detected. Buy order placed.")
                        else:
                            main_logger.info("Buy Order Already Exists")
                            
                    elif check_trend(SYMBOL, mt5.TIMEFRAME_M5) == 0:
                        if result['sell'] == False:
                            main_logger.info("Downtrend detected. Placing sell order.")
                            market_order(SYMBOL, VOLUME, "sell", sl_pips=STOPLOSS, tp_pips=TAKEPROFIT)
                            trade_logger.info("Downtrend detected. Sell order placed.")
                        else:
                            main_logger.info("Sell Order Already Exists")
                else:
                    main_logger.info(f"Entry logic not triggered for {SYMBOL}. No Crossovers")
                
            else:
                main_logger.debug("No new tick detected.")

            # Sleep to prevent overloading
            time.sleep(1)

        except Exception as e:
            main_logger.error(f"Error in main loop: {e}")
            error_logger.error(f"Error in main loop: {e}")


# Entry point
if __name__ == "__main__":
    initialize_loggers()
    main_trading_loop()


KeyboardInterrupt: 

In [ ]:
import Me